# Module 01: NumPy for Machine Learning
## Notebook 03: Vectorization, Broadcasting Rules, and Advanced Array Operations

Vectorization and broadcasting are the core engines of numerical computing in Python. By eliminating explicit Python `for` loops, operations execute directly at compiled C speed and scale seamlessly to millions of data points and high-dimensional tensors.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Explain the architectural mechanics of **vectorization** and benchmark its speedup over loops.
2. Utilize universal functions (**ufuncs**) for mathematical transformations and gradient clipping.
3. Master the **3 General Broadcasting Rules** and anticipate resulting array shapes.
4. Perform conditional filtering and boolean masking using `np.where()` and `np.select()`.
5. Extract top predictions and ranking indices using `np.argmax()` and `np.argsort()`.
6. Implement a **Numerically Stable Multi-Dimensional Softmax** function using broadcasting and `keepdims=True`.
7. Execute **Loop-less One-Hot Encoding** using identity matrix indexing.
8. Compute **Batched Mahalanobis Distances** across feature matrices without nested loops.

In [ ]:
import numpy as np
import time

print(f"NumPy version: {np.__version__}")

### 1. The Vectorization Advantage

In standard Python, executing a loop requires the Python interpreter to inspect the type of every single object at each iteration, check method tables, and handle dynamic dispatch.

In NumPy:
- Operations execute over contiguous memory blocks using compiled C/Fortran routines.
- SIMD (**Single Instruction, Multiple Data**) processor instructions execute arithmetic operations across multiple array elements simultaneously.

In [ ]:
# Simulating Euclidean Distance between two feature vectors of 2,000,000 values
N = 2_000_000
u = np.random.rand(N)
v = np.random.rand(N)

# 1. Loop-based computation
start = time.time()
dist_loop = 0.0
for i in range(N):
    diff = u[i] - v[i]
    dist_loop += diff * diff
dist_loop = dist_loop ** 0.5
loop_time = time.time() - start

# 2. Vectorized computation
start = time.time()
dist_vec = np.sqrt(np.sum((u - v) ** 2))
vec_time = time.time() - start

print(f"Loop Distance:         {dist_loop:.4f} (took {loop_time:.4f}s)")
print(f"Vectorized Distance:   {dist_vec:.4f} (took {vec_time:.4f}s)")
print(f"Vectorization Speedup: {loop_time / vec_time:.1f}x faster!")

---
### 2. Universal Functions (ufuncs)

A universal function (**ufunc**) operates on `ndarray`s element-by-element.
Essential ufuncs in Machine Learning:
- Exponential & Logarithm: `np.exp()` (softmax / sigmoid), `np.log()`, `np.log1p()` (log loss, entropy).
- Power & Root: `np.sqrt()`, `np.square()`, `np.power()` (MSE, RMSE).
- Clipping & Clamping: `np.clip()` (preventing numerical explosion and exploding gradients).

In [ ]:
logits = np.array([-10.0, -2.5, 0.0, 2.5, 10.0, 100.0])

# Sigmoid activation function: 1 / (1 + exp(-z))
# Using np.clip to prevent overflow encountered in exp
clipped_logits = np.clip(logits, -50.0, 50.0)
sigmoid_probs = 1.0 / (1.0 + np.exp(-clipped_logits))

print("Original Logits:       ", logits)
print("Sigmoid Probabilities: ", np.round(sigmoid_probs, 4))

# Safe log loss evaluation with np.clip
predictions = np.array([0.999, 0.001, 0.85, 0.40])
safe_preds = np.clip(predictions, 1e-15, 1 - 1e-15)
log_loss = -np.log(safe_preds)
print("\nCross-Entropy Losses:   ", np.round(log_loss, 4))

---
### 3. The 3 General Broadcasting Rules

Broadcasting describes how NumPy treats arrays with different shapes during arithmetic operations.

#### The Rules:
1. **Right-Alignment**: Compare the dimensions of both arrays starting from the **trailing (rightmost)** dimension and work backward.
2. **Compatibility**: Two dimensions are compatible if:
   - They are equal, OR
   - One of them is 1.
3. **Expansion**: Dimensions of size 1 are virtually stretched (without copying memory) to match the larger dimension. If dimensions do not match and neither is 1, NumPy raises a `ValueError: operands could not be broadcast together`.

```text
Example 1: (3, 4) + (4,)
   Array A:   3  x  4
   Array B:         4  (padded to 1 x 4)
   Result:    3  x  4  (Valid!)

Example 2: (4, 1) * (1, 5)
   Array A:   4  x  1
   Array B:   1  x  5
   Result:    4  x  5  (Outer product!)

Example 3: (3, 4) + (3,)  --> INCOMPATIBLE!
   Array A:   3  x  4
   Array B:         3  (4 != 3 and neither is 1 -> FAILS)
   Fix: Reshape B to (3, 1)!
```

In [ ]:
# Example 1: Subtracting feature means from dataset X (centering)
# Dataset: 4 samples, 3 features
X = np.array([
    [10.0, 20.0, 30.0],
    [12.0, 24.0, 32.0],
    [14.0, 22.0, 28.0],
    [16.0, 26.0, 34.0]
])

feature_means = np.mean(X, axis=0) # Shape: (3,)

print(f"X shape:             {X.shape}")
print(f"feature_means shape: {feature_means.shape}")

# Broadcasting (4, 3) - (3,) -> feature_means is broadcast across all 4 rows
X_centered = X - feature_means
print("\nMean-centered feature matrix:\n", X_centered)

In [ ]:
# Example 2: Outer product using (M, 1) and (1, N)
# Compute pairwise multiplication grid
u = np.array([1, 2, 3])[:, np.newaxis]  # Shape: (3, 1)
v = np.array([10, 20, 30, 40])[np.newaxis, :]  # Shape: (1, 4)

grid = u * v  # Broadcasts to (3, 4)
print("u shape:", u.shape)
print("v shape:", v.shape)
print("Broadcasted product grid (3, 4):\n", grid)

---
### 4. Boolean Masking and Conditional Filtering

Boolean masking allows you to select, count, and modify elements based on complex conditional logic without writing loops.
- Relational operators: `>`, `<`, `>=`, `<=`, `==`, `!=`
- Logical bitwise operators: `&` (AND), `|` (OR), `~` (NOT). *Parentheses around each sub-clause are mandatory!*
- `np.where(condition, value_if_true, value_if_false)`: Vectorized ternary operator.
- `np.select(conditions_list, choice_list, default)`: Multi-way branching.

In [ ]:
scores = np.array([45, 88, 72, 95, 30, 60, 82, 91])

# 1. Creating a boolean mask
passing_mask = scores >= 60
print("Passing mask:       ", passing_mask)

# 2. Filtering with boolean indexing
passing_scores = scores[passing_mask]
print("Passing scores only:", passing_scores)

# 3. Compound condition: High performers (scores between 80 and 95 inclusive)
elite_mask = (scores >= 80) & (scores <= 95)
print("Elite scores (80-95):", scores[elite_mask])

# 4. np.where: Labeling Pass vs Fail
labels = np.where(scores >= 60, "PASS", "FAIL")
print("Pass/Fail labels:   ", labels)

# 5. Multi-way grading using np.select
conditions = [
    scores >= 90,
    (scores >= 75) & (scores < 90),
    (scores >= 60) & (scores < 75)
]
grades = ["A", "B", "C"]
assigned_grades = np.select(conditions, grades, default="F")
print("Letter grades:      ", assigned_grades)

---
### 5. Searching and Sorting Operations

In machine learning, we constantly need to:
- Find the class with highest probability (`np.argmax()`).
- Rank predictions or find the $K$ nearest neighbors (`np.argsort()`).
- Check if all data meets quality criteria (`np.all()`, `np.any()`).

In [ ]:
# Simulated Softmax Output for 4 samples across 3 classes (Class 0: Cat, Class 1: Dog, Class 2: Bird)
probs = np.array([
    [0.10, 0.70, 0.20],  # Sample 0 -> Dog
    [0.85, 0.05, 0.10],  # Sample 1 -> Cat
    [0.15, 0.25, 0.60],  # Sample 2 -> Bird
    [0.30, 0.40, 0.30]   # Sample 3 -> Dog
])

# Argmax along axis 1 (across columns) gives the winning class index for each sample
predicted_classes = np.argmax(probs, axis=1)
confidence_scores = np.max(probs, axis=1)

class_names = np.array(["Cat", "Dog", "Bird"])
print("Predicted class indices: ", predicted_classes)
print("Predicted class names:   ", class_names[predicted_classes])
print("Confidence scores:       ", confidence_scores)

# Argsort for ranking feature importances (ascending order)
feature_importances = np.array([0.12, 0.45, 0.03, 0.28, 0.12])
sorted_indices = np.argsort(feature_importances)[::-1] # descending
print("\nFeature ranking from most to least important:", sorted_indices)

---
### 6. Advanced Usages: Numerically Stable Softmax, One-Hot Encoding, and Mahalanobis Distance

#### A. Numerically Stable Multi-Dimensional Softmax (The Log-Sum-Exp Trick)

In multi-class classification and Transformer attention heads:
$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}$$

**The Numerical Instability Problem:**
In 64-bit float, $e^{710} = \infty$ (overflow). If logits contain large values (e.g. $[1000, 1001, 1002]$), standard softmax evaluates to $\frac{\infty}{\infty} = \text{NaN}$!
**The Mathematical Fix:**
Subtract the maximum logit from all elements before exponentiation:
$$\text{Softmax}(z_i) = \frac{e^{z_i - \max(z)}}{\sum_j e^{z_j - \max(z)}}$$
Because $\frac{e^{z_i - C}}{\sum e^{z_j - C}} = \frac{e^{z_i} e^{-C}}{e^{-C} \sum e^{z_j}} = \text{Softmax}(z_i)$, the result is mathematically identical, but the maximum exponent is now $e^0 = 1.0$, completely eliminating overflow!

In [ ]:
def softmax_stable(logits, axis=-1):
    # Subtract max along target axis with keepdims=True for broadcasting
    max_logits = np.max(logits, axis=axis, keepdims=True)
    exp_shifted = np.exp(logits - max_logits)
    sum_exp = np.sum(exp_shifted, axis=axis, keepdims=True)
    return exp_shifted / sum_exp

# Test on extreme logits that would cause standard exp() to overflow to NaN
extreme_logits = np.array([
    [1000.0, 1002.0, 999.0],   # Row 0
    [-500.0, -498.0, -502.0]   # Row 1
])

stable_probs = softmax_stable(extreme_logits, axis=-1)
print("Softmax Probabilities for Extreme Logits:\n", np.round(stable_probs, 4))
print("Sum of probabilities along rows (should be exactly 1.0):", np.sum(stable_probs, axis=-1))

#### B. Loop-less Vectorized One-Hot Encoding

Given a 1D vector of discrete class labels $y \in \{0, 1, \dots, C-1\}$, how do you convert it into a 2D binary indicator matrix of shape $(N, C)$ with zero loops?
By indexing an identity matrix $I_C$ using the label vector:
$$Y_{one\_hot} = I_C[y]$$

In [ ]:
labels = np.array([0, 2, 1, 3, 0, 2])
num_classes = 4

# Zero-loop vectorized One-Hot Encoding via identity indexing
one_hot_matrix = np.eye(num_classes, dtype=np.float32)[labels]

print("Original Integer Labels:\n", labels)
print("\nVectorized One-Hot Encoded Matrix (Shape: {}):\n".format(one_hot_matrix.shape), one_hot_matrix)

#### C. Vectorized Batched Mahalanobis Distance

Euclidean distance assumes feature axes are independent and have equal variance.
The **Mahalanobis Distance** accounts for the covariance between features:
$$D_M(u, v) = \sqrt{(u - v)^T \Sigma^{-1} (u - v)}$$

We can compute this for an entire batch of query points against a reference distribution simultaneously using broadcasting and matrix contractions without any loops.

In [ ]:
# Reference distribution covariance matrix (2 features)
cov_matrix = np.array([[2.0, 0.8], [0.8, 1.5]])
inv_cov = np.linalg.inv(cov_matrix)
mean_vector = np.array([10.0, 20.0])

# Batch of 4 query points
X_query = np.array([
    [10.0, 20.0],  # Centroid -> Distance should be 0.0
    [12.0, 21.0],
    [8.0, 19.0],
    [15.0, 25.0]
])

# Difference vectors: shape (4, 2)
diff = X_query - mean_vector

# Vectorized quadratic form: diag(diff @ inv_cov @ diff.T)
# Computed efficiently via element-wise multiplication and row sum:
mahalanobis_sq = np.sum((diff @ inv_cov) * diff, axis=1)
mahalanobis_dist = np.sqrt(np.maximum(mahalanobis_sq, 0.0))

print("Query Points:\n", X_query)
print("\nVectorized Mahalanobis Distances from Centroid:", np.round(mahalanobis_dist, 4))

### Summary & Next Steps
In this notebook, you mastered:
- The performance superiority of vectorization over interpreter loops.
- Universal functions (`ufuncs`) and safe numerical transformations.
- The 3 rules of broadcasting and debugging dimension mismatches.
- Conditional indexing, `np.where()`, and `np.select()`.
- **Numerically stable multi-dimensional Softmax** preventing exponential overflow.
- **Loop-less One-Hot Encoding** with identity matrix indexing.
- **Vectorized Mahalanobis Distance** accounting for feature covariance.

**Next Notebook:** `04_math_stats_and_linear_algebra.ipynb` — SVD for image compression, Cholesky decomposition, and building Principal Component Analysis (PCA) from scratch.